# Voice Benchmark: OmniVoice vs ElevenLabs

**Muc dich:** So sanh chat luong giong doc giua OmniVoice (Colab T4) va ElevenLabs (chuẩn vàng).

**Quy trinh:**
1. Cai dat dependencies + kiem tra GPU
2. Load voice packs (7 giong mau tu ElevenLabs + noi dung text)
3. Sinh audio bang OmniVoice voi cung text + voice sample
4. So sanh waveform (song am)
5. So sanh spectrogram + pitch contour
6. Tinh quality metrics (spectral, MFCC, energy)
7. Bang tong hop + ket luan

**Repo:** github.com/doanquangkien/voice-notebooks (PUBLIC)

---

## Cell 1: Cai dat + GPU Check

In [ ]:
# Cell 1: Install dependencies + GPU check
print('Dang cai dat (~2 phut)...')
!pip install -q omnivoice gradio "numpy<2.1" "requests==2.32.4" "torch<2.7"
!pip install -q librosa matplotlib soundfile ipython
print('Cai dat hoan tat!')

# GPU check
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu} ({vram:.1f} GB)')
else:
    raise RuntimeError('KHONG TIM THAY GPU! Bat GPU trong Runtime > Change runtime type')

## Cell 2: Load Voice Packs

In [ ]:
# Cell 2: Load voice packs — download tu voice-notebooks repo
import os, json, urllib.request

REPO_BASE = 'https://raw.githubusercontent.com/doanquangkien/voice-notebooks/main/analysis/voice_packs'

# Dinh nghia 7 giong: (key, audio_file, txt_file)
VOICES = [
    ('nam-cong-nghe',    'Nam_CN.mp3',           'Nam_CN.txt'),
    ('minh-anh',         'Minh_Anh.mp3',         'Minh_Anh.txt'),
    ('thanh-nien-tu-tin','Thanh_Nien_Tu_Tin.mp3','Thanh_Nien_Tu_Tin.txt'),
    ('nam-tram-am',      'XL_Nam_Tram_Am.mp3',   'XL_Nam_Tram_Am.txt'),
    ('adam',             'ADAM.wav',             'ADAM.txt'),
    ('ngoc-huyen',       'Ngoc_Huyen.mp3',       'Ngoc_Huyen.txt'),
    ('nho-ngot-ngao',    'NhoNgotNgao.mp3',      'NhoNgotNgao.txt'),
]

os.makedirs('voice_packs', exist_ok=True)
voice_data = []

for key, audio_file, txt_file in VOICES:
    # Download audio
    audio_path = f'voice_packs/{audio_file}'
    if not os.path.exists(audio_path):
        url = f'{REPO_BASE}/{audio_file}'
        print(f'Downloading {audio_file}...')
        urllib.request.urlretrieve(url, audio_path)
    
    # Download text
    txt_path = f'voice_packs/{txt_file}'
    if not os.path.exists(txt_path):
        url = f'{REPO_BASE}/{txt_file}'
        urllib.request.urlretrieve(url, txt_path)
    
    with open(txt_path, 'r', encoding='utf-8') as f:
        text = f.read().strip()
    
    voice_data.append({
        'key': key,
        'audio_path': audio_path,
        'text': text,
        'audio_file': audio_file,
    })
    print(f'[{key}] {len(text)} chars — {audio_file}')

print(f'\nLoaded {len(voice_data)} voices')

## Cell 3: Generate OmniVoice Audio

In [ ]:
# Cell 3: Generate OmniVoice audio cho tat ca 7 giong
# Shim AutoFeatureExtractor cho transformers 5.x
import transformers as _tf
class _SafeAutoFeatureExtractor:
    @staticmethod
    def from_pretrained(model_name, **kwargs):
        try:
            from transformers import AutoConfig
            cfg = AutoConfig.from_pretrained(model_name, trust_remote_code=True, **kwargs)
            sr = getattr(cfg, 'sampling_rate', 24000)
        except Exception:
            sr = 24000
        class _Result:
            sampling_rate = sr
        return _Result()
_tf.AutoFeatureExtractor = _SafeAutoFeatureExtractor

# Load OmniVoice model
from omnivoice import OmniVoice
import torchaudio
import torch

print('Loading OmniVoice model...')
model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', torch_dtype=torch.float16)
model = model.to('cuda')
print('Model loaded!')

# Default config (baseline)
CONFIG = {
    'steps': 32,
    'guidance_scale': 1.8,
    'speed': 0.95,
}

os.makedirs('output_omnivoice', exist_ok=True)

for vd in voice_data:
    key = vd['key']
    ref_audio = vd['audio_path']
    text = vd['text']
    output_path = f'output_omnivoice/{key}.wav'
    
    print(f'\n--- Generating: {key} ---')
    print(f'Text: {text[:80]}...')
    
    # Create voice clone prompt
    voice_prompt = model.create_voice_clone_prompt(ref_audio=ref_audio)
    
    # Generate audio
    audio = model.generate(
        text=text,
        voice_prompt=voice_prompt,
        steps=CONFIG['steps'],
        guidance_scale=CONFIG['guidance_scale'],
        speed=CONFIG['speed'],
        language='vi',
    )
    
    # Save output
    torchaudio.save(output_path, audio.cpu(), 24000)
    print(f'Saved: {output_path} ({audio.shape[-1]/24000:.1f}s)')

print(f'\nHoan tat! {len(voice_data)} audio files trong output_omnivoice/')

## Cell 4: Waveform Comparison

In [ ]:
# Cell 4: Waveform comparison — song am ElevenLabs vs OmniVoice
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf

def load_audio(path, sr=24000):
    """Load audio file, resample to target sr"""
    y, orig_sr = librosa.load(path, sr=sr)
    return y, sr

fig, axes = plt.subplots(len(voice_data), 2, figsize=(16, 3.5 * len(voice_data)))
fig.suptitle('Waveform Comparison: ElevenLabs (trai) vs OmniVoice (phai)', fontsize=14, fontweight='bold')

for i, vd in enumerate(voice_data):
    key = vd['key']
    
    # Load ElevenLabs sample
    y_el, sr_el = load_audio(vd['audio_path'])
    
    # Load OmniVoice output
    ov_path = f'output_omnivoice/{key}.wav'
    y_ov, sr_ov = load_audio(ov_path)
    
    # Plot ElevenLabs
    ax_el = axes[i, 0]
    librosa.display.waveshow(y_el, sr=sr_el, ax=ax_el, color='#2196F3')
    ax_el.set_title(f'{key} — ElevenLabs ({len(y_el)/sr_el:.1f}s)', fontsize=10)
    ax_el.set_ylabel('Amplitude')
    ax_el.set_xlim(0, max(len(y_el)/sr_el, len(y_ov)/sr_ov))
    
    # Plot OmniVoice
    ax_ov = axes[i, 1]
    librosa.display.waveshow(y_ov, sr=sr_ov, ax=ax_ov, color='#FF9800')
    ax_ov.set_title(f'{key} — OmniVoice ({len(y_ov)/sr_ov:.1f}s)', fontsize=10)
    ax_ov.set_ylabel('Amplitude')
    ax_ov.set_xlim(0, max(len(y_el)/sr_el, len(y_ov)/sr_ov))

plt.tight_layout()
plt.savefig('comparison_waveform.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: comparison_waveform.png')

## Cell 5: Spectrogram + Pitch Contour

In [ ]:
# Cell 5: Spectrogram + Pitch contour comparison
fig, axes = plt.subplots(len(voice_data), 4, figsize=(20, 3.5 * len(voice_data)))
fig.suptitle('Spectrogram + Pitch: ElevenLabs (trai) vs OmniVoice (phai)', fontsize=14, fontweight='bold')

for i, vd in enumerate(voice_data):
    key = vd['key']
    
    y_el, sr_el = load_audio(vd['audio_path'])
    y_ov, sr_ov = load_audio(f'output_omnivoice/{key}.wav')
    
    # Mel Spectrogram — ElevenLabs
    S_el = librosa.feature.melspectrogram(y=y_el, sr=sr_el, n_mels=80)
    S_el_db = librosa.power_to_db(S_el, ref=np.max)
    ax = axes[i, 0]
    librosa.display.specshow(S_el_db, sr=sr_el, x_axis='time', y_axis='mel', ax=ax, cmap='magma')
    ax.set_title(f'{key} — Mel Spectrogram (EL)', fontsize=9)
    
    # Mel Spectrogram — OmniVoice
    S_ov = librosa.feature.melspectrogram(y=y_ov, sr=sr_ov, n_mels=80)
    S_ov_db = librosa.power_to_db(S_ov, ref=np.max)
    ax = axes[i, 1]
    librosa.display.specshow(S_ov_db, sr=sr_ov, x_axis='time', y_axis='mel', ax=ax, cmap='magma')
    ax.set_title(f'{key} — Mel Spectrogram (OV)', fontsize=9)
    
    # Pitch Contour — ElevenLabs
    f0_el = librosa.yin(y_el, fmin=60, fmax=400, sr=sr_el)
    times_el = librosa.times_like(f0_el, sr=sr_el)
    ax = axes[i, 2]
    ax.plot(times_el, f0_el, color='#2196F3', linewidth=0.8)
    ax.set_title(f'{key} — Pitch (EL)', fontsize=9)
    ax.set_ylabel('Hz')
    ax.set_ylim(60, 400)
    ax.grid(True, alpha=0.3)
    
    # Pitch Contour — OmniVoice
    f0_ov = librosa.yin(y_ov, fmin=60, fmax=400, sr=sr_ov)
    times_ov = librosa.times_like(f0_ov, sr=sr_ov)
    ax = axes[i, 3]
    ax.plot(times_ov, f0_ov, color='#FF9800', linewidth=0.8)
    ax.set_title(f'{key} — Pitch (OV)', fontsize=9)
    ax.set_ylabel('Hz')
    ax.set_ylim(60, 400)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_spectrogram_pitch.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: comparison_spectrogram_pitch.png')

## Cell 6: Quality Metrics

In [ ]:
# Cell 6: Quality metrics — phan tich khach quan
import pandas as pd

def compute_metrics(y, sr):
    """Tinh cac audio features"""
    # RMS Energy
    rms = np.sqrt(np.mean(y**2))
    
    # Spectral Centroid (do sang cua am thanh)
    sc = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    
    # Spectral Bandwidth (do rong bang tan so)
    sb = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
    
    # Zero Crossing Rate (do nhieu / tan so)
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))
    
    # MFCC (dac trung am sac)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)
    
    # Pitch statistics
    f0 = librosa.yin(y, fmin=60, fmax=400, sr=sr)
    f0_voiced = f0[f0 > 60]  # Chi lay pitch co am
    pitch_mean = np.mean(f0_voiced) if len(f0_voiced) > 0 else 0
    pitch_std = np.std(f0_voiced) if len(f0_voiced) > 0 else 0
    
    # Duration
    duration = len(y) / sr
    
    return {
        'duration_s': round(duration, 2),
        'rms_energy': round(rms, 4),
        'spectral_centroid_hz': round(sc, 1),
        'spectral_bandwidth_hz': round(sb, 1),
        'zero_crossing_rate': round(zcr, 4),
        'pitch_mean_hz': round(pitch_mean, 1),
        'pitch_std_hz': round(pitch_std, 1),
        'mfcc_0': round(mfcc_mean[0], 2),
        'mfcc_1': round(mfcc_mean[1], 2),
        'mfcc_2': round(mfcc_mean[2], 2),
    }

# Tinh metrics cho tat ca
rows = []
for vd in voice_data:
    key = vd['key']
    
    y_el, sr_el = load_audio(vd['audio_path'])
    y_ov, sr_ov = load_audio(f'output_omnivoice/{key}.wav')
    
    m_el = compute_metrics(y_el, sr_el)
    m_ov = compute_metrics(y_ov, sr_ov)
    
    rows.append({
        'voice': key,
        'source': 'ElevenLabs',
        **m_el
    })
    rows.append({
        'voice': key,
        'source': 'OmniVoice',
        **m_ov
    })

df = pd.DataFrame(rows)

# Hien thi bang
print('=' * 100)
print('VOICE QUALITY METRICS — ElevenLabs vs OmniVoice')
print('=' * 100)
print(df.to_string(index=False))

# Tinh delta
print('\n' + '=' * 100)
print('DELTA (OmniVoice - ElevenLabs) — Am = it hon, Duong = nhieu hon')
print('=' * 100)

delta_rows = []
for vd in voice_data:
    key = vd['key']
    el = df[(df['voice'] == key) & (df['source'] == 'ElevenLabs')].iloc[0]
    ov = df[(df['voice'] == key) & (df['source'] == 'OmniVoice')].iloc[0]
    
    delta = {'voice': key}
    for col in ['duration_s', 'rms_energy', 'spectral_centroid_hz', 'spectral_bandwidth_hz',
                'zero_crossing_rate', 'pitch_mean_hz', 'pitch_std_hz']:
        d = ov[col] - el[col]
        delta[col] = round(d, 3)
    delta_rows.append(delta)

df_delta = pd.DataFrame(delta_rows)
print(df_delta.to_string(index=False))

# Save
df.to_csv('metrics_comparison.csv', index=False)
df_delta.to_csv('metrics_delta.csv', index=False)
print('\nSaved: metrics_comparison.csv, metrics_delta.csv')

## Cell 7: Summary + Parameter Recommendations

In [ ]:
# Cell 7: Tong hop + de xuat config
print('=' * 80)
print('TONG HOP KET QUA BENCHMARK')
print('=' * 80)

# Tinh trung binh delta
numeric_cols = ['duration_s', 'rms_energy', 'spectral_centroid_hz', 'spectral_bandwidth_hz',
                'zero_crossing_rate', 'pitch_mean_hz', 'pitch_std_hz']
avg_delta = df_delta[numeric_cols].mean()

print(f'\nConfig hien tai: steps={CONFIG["steps"]}, guidance_scale={CONFIG["guidance_scale"]}, speed={CONFIG["speed"]}')
print(f'\nTrung binh delta (OmniVoice - ElevenLabs):')
for col in numeric_cols:
    d = avg_delta[col]
    sign = '+' if d > 0 else ''
    unit = 's' if 'duration' in col else 'Hz' if 'hz' in col.lower() or 'pitch' in col else ''
    print(f'  {col:30s}: {sign}{d:.3f} {unit}')

print(f'\n--- PHAN TICH ---')
print(f'1. Duration: OmniVoice am/dai hon ElevenLabs?')
print(f'   → Neu am hon: speed dang lam cham qua. Tang speed len.')
print(f'   → Neu dai hon: speed dang lam nhanh qua. Giam speed xuong.')
print(f'\n2. Pitch Mean: OmniVoice cao/thap hon?')
print(f'   → Cao hon: giong bi "keo" len. Co the do guidance_scale qua cao.')
print(f'   → Thap hon: giong bi "day" xuong. Tang guidance_scale.')
print(f'\n3. Pitch Std (do dao dong pitch):')
print(f'   → Thap hon = giong "phang", it cam xuc. Tang guidance_scale hoac steps.')
print(f'   → Cao hon = giong "nhay", qua nhieu dao dong.')
print(f'\n4. Spectral Centroid: do "sang" cua giong')
print(f'   → Cao hon = giong sang hon, tre hon.')
print(f'   → Thap hon = giong tram hon, gia hon.')

print(f'\n--- DE XUAT CONFIG TIEP THEO ---')
print(f'Chay lai Cell 3 voi cac config sau de so sanh:')
print(f'')
print(f'CONFIG_A = {{"steps": 48, "guidance_scale": 1.8, "speed": 0.95}}  # Nhieu steps hon')
print(f'CONFIG_B = {{"steps": 32, "guidance_scale": 2.5, "speed": 0.95}}  # Cao guidance')
print(f'CONFIG_C = {{"steps": 32, "guidance_scale": 1.8, "speed": 0.85}}  # Cham hon')
print(f'CONFIG_D = {{"steps": 48, "guidance_scale": 2.5, "speed": 0.90}}  # Combo')
print(f'')
print(f'Sau khi chay 4 configs → doi chieu metrics → chon config tot nhat.')

print(f'\n' + '=' * 80)
print(f'KET THUC BENCHMARK')
print(f'Files output:')
print(f'  - output_omnivoice/*.wav      (7 audio OmniVoice)')
print(f'  - comparison_waveform.png     (bieu do song am)')
print(f'  - comparison_spectrogram_pitch.png (bieu do tan so + pitch)')
print(f'  - metrics_comparison.csv      (bang so sanh)')
print(f'  - metrics_delta.csv           (bang delta)')
print(f'=' * 80)

## Cell 8: Zip + Download All Results

In [ ]:
# Cell 8: Zip tat ca output + download 1 file
import zipfile
from google.colab import files as colab_files

zip_name = 'voice_benchmark_results.zip'

# Tat ca files can zip
output_files = []

# Audio OmniVoice
for vd in voice_data:
    wav_path = f'output_omnivoice/{vd["key"]}.wav'
    if os.path.exists(wav_path):
        output_files.append(wav_path)

# Charts
for f in ['comparison_waveform.png', 'comparison_spectrogram_pitch.png']:
    if os.path.exists(f):
        output_files.append(f)

# CSV
for f in ['metrics_comparison.csv', 'metrics_delta.csv']:
    if os.path.exists(f):
        output_files.append(f)

# Tao zip
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in output_files:
        zf.write(fpath)
        print(f'  + {fpath}')

zip_size = os.path.getsize(zip_name) / 1024 / 1024
print(f'\n{zip_name}: {zip_size:.1f} MB ({len(output_files)} files)')

# Tu dong download
print('\nDang tai xuong...')
colab_files.download(zip_name)
print('Hoan tat!')